In [ ]:
using System;
using System.Collections.Generic;

public enum OrderStatus
{
    Created,
    InProgress,
    Completed,
    Cancelled
}

public delegate void OrderNotificationHandler(Order order, OrderStatus oldStatus, OrderStatus newStatus);

public abstract class Person
{
    private string firstName;
    private string lastName;

    public Person(string firstName, string lastName)
    {
        this.firstName = firstName;
        this.lastName = lastName;
    }

    public string FirstName
    {
        get { return firstName; }
        set { firstName = value; }
    }

    public string LastName
    {
        get { return lastName; }
        set { lastName = value; }
    }

    public virtual string GetFullName()
    {
        return FirstName + " " + LastName;
    }

    public abstract string GetInfo();
}

public class Customer : Person
{
    public string PhoneNumber;

    public Customer(string firstName, string lastName, string phoneNumber)
        : base(firstName, lastName)
    {
        PhoneNumber = phoneNumber;
    }

    public void UpdatePhoneNumber(string newNumber)
    {
        PhoneNumber = newNumber;
    }

    public override string GetInfo()
    {
        return "Заказчик: " + GetFullName() + " | Телефон: " + PhoneNumber;
    }

    public override string ToString()
    {
        return GetInfo();
    }
}

public class Employee : Person
{
    public string Position;
    public int OrdersProcessed;

    public Employee(string firstName, string lastName, string position)
        : base(firstName, lastName)
    {
        Position = position;
        OrdersProcessed = 0;
    }

    public void OnOrderStatusChanged(Order order, OrderStatus oldStatus, OrderStatus newStatus)
    {
        Console.WriteLine("[Уведомление для " + GetFullName() + "]: заказ #" + order.OrderId +
            " \"" + order.Description + "\" сменил статус: " + oldStatus + " -> " + newStatus);
    }

    public void IncrementOrdersProcessed()
    {
        OrdersProcessed++;
    }

    public void ResetOrdersProcessed()
    {
        OrdersProcessed = 0;
    }

    public override string GetInfo()
    {
        return "Сотрудник: " + GetFullName() + " | Должность: " + Position +
            " | Обработано заказов: " + OrdersProcessed;
    }

    public override string ToString()
    {
        return GetInfo();
    }
}

public class Order
{
    private static int nextId = 1;

    public int OrderId;
    public string Description;
    public Customer Customer;
    public Employee AssignedEmployee;
    public OrderStatus Status;
    public DateTime CreatedAt;

    public event OrderNotificationHandler StatusChanged;

    public Order(string description, Customer customer)
    {
        OrderId = nextId;
        nextId++;
        Description = description;
        Customer = customer;
        Status = OrderStatus.Created;
        CreatedAt = DateTime.Now;
    }

    public void AssignEmployee(Employee employee)
    {
        if (AssignedEmployee != null)
        {
            StatusChanged -= AssignedEmployee.OnOrderStatusChanged;
        }

        AssignedEmployee = employee;
        StatusChanged += employee.OnOrderStatusChanged;
    }

    public void ChangeStatus(OrderStatus newStatus)
    {
        if (Status == newStatus)
        {
            return;
        }

        OrderStatus oldStatus = Status;
        Status = newStatus;

        if (StatusChanged != null)
        {
            StatusChanged(this, oldStatus, newStatus);
        }

        if (newStatus == OrderStatus.Completed && AssignedEmployee != null)
        {
            AssignedEmployee.IncrementOrdersProcessed();
        }
    }

    public string GetOrderInfo()
    {
        string employeeName = "Не назначен";
        if (AssignedEmployee != null)
        {
            employeeName = AssignedEmployee.GetFullName();
        }

        return "Заказ #" + OrderId + " [" + Status + "] от " + CreatedAt.ToString("dd.MM.yyyy HH:mm") +
            " | \"" + Description + "\" | Клиент: " + Customer.GetFullName() +
            " | Ответственный: " + employeeName;
    }

    public override string ToString()
    {
        return GetOrderInfo();
    }
}

public class Company
{
    public string Name;
    public List<Employee> Employees = new List<Employee>();
    public List<Customer> Customers = new List<Customer>();
    public List<Order> Orders = new List<Order>();

    public Company(string name)
    {
        Name = name;
    }

    public void AddEmployee(Employee employee)
    {
        Employees.Add(employee);
    }

    public void AddCustomer(Customer customer)
    {
        Customers.Add(customer);
    }

    public Order CreateOrder(string description, Customer customer)
    {
        Order order = new Order(description, customer);
        Orders.Add(order);

        if (!Customers.Contains(customer))
        {
            Customers.Add(customer);
        }
        return order;
    }

    public Order FindOrderById(int orderId)
    {
        foreach (Order order in Orders)
        {
            if (order.OrderId == orderId)
            {
                return order;
            }
        }
        return null;
    }

    public void AssignEmployeeToOrder(int orderId, Employee employee)
    {
        Order order = FindOrderById(orderId);
        if (order != null)
        {
            order.AssignEmployee(employee);
        }
    }

    public void ChangeOrderStatus(int orderId, OrderStatus newStatus)
    {
        Order order = FindOrderById(orderId);
        if (order != null)
        {
            order.ChangeStatus(newStatus);
        }
    }

    public List<Order> GetOrdersByStatus(OrderStatus status)
    {
        List<Order> result = new List<Order>();
        foreach (Order order in Orders)
        {
            if (order.Status == status)
            {
                result.Add(order);
            }
        }
        return result;
    }

    public List<Order> FindOrdersByCustomer(Customer customer)
    {
        List<Order> result = new List<Order>();
        foreach (Order order in Orders)
        {
            if (order.Customer == customer)
            {
                result.Add(order);
            }
        }
        return result;
    }

    public void PrintOrders(List<Order> orders)
    {
        if (orders.Count == 0)
        {
            Console.WriteLine("  (нет заказов)");
        }
        foreach (Order order in orders)
        {
            Console.WriteLine("  " + order.GetOrderInfo());
        }
    }

    public void PrintEmployeeReport()
    {
        Console.WriteLine("Отчет по сотрудникам компании \"" + Name + "\":");
        foreach (Employee employee in Employees)
        {
            Console.WriteLine("- " + employee.GetFullName() + " (" + employee.Position +
                "): обработано заказов — " + employee.OrdersProcessed);
        }
    }
}


Company company = new Company("ООО Райский уголок");

Employee ivan = new Employee("Иван", "Петров", "Менеджер");
Employee anna = new Employee("Анна", "Сидорова", "Инженер");
company.AddEmployee(ivan);
company.AddEmployee(anna);

Customer oleg = new Customer("Олег", "Смирнов", "+7-900-111-22-33");
Customer maria = new Customer("Мария", "Кузнецова", "+7-900-444-55-66");
company.AddCustomer(oleg);
company.AddCustomer(maria);

Console.WriteLine("=== Участники ===");
Console.WriteLine(ivan);
Console.WriteLine(anna);
Console.WriteLine(oleg);
Console.WriteLine(maria);


Order order1 = company.CreateOrder("Разработка сайта", oleg);
Order order2 = company.CreateOrder("Настройка сети", maria);
Order order3 = company.CreateOrder("Ремонт сервера", oleg);

Console.WriteLine();
Console.WriteLine("=== Созданные заказы ===");
company.PrintOrders(company.Orders);


company.AssignEmployeeToOrder(order1.OrderId, ivan);
company.AssignEmployeeToOrder(order2.OrderId, anna);
company.AssignEmployeeToOrder(order3.OrderId, ivan);


Console.WriteLine();
Console.WriteLine("=== Изменение статусов ===");
company.ChangeOrderStatus(order1.OrderId, OrderStatus.InProgress);
company.ChangeOrderStatus(order2.OrderId, OrderStatus.InProgress);
company.ChangeOrderStatus(order1.OrderId, OrderStatus.Completed);
company.ChangeOrderStatus(order3.OrderId, OrderStatus.Cancelled);


Console.WriteLine();
Console.WriteLine("=== Переназначение заказа #2 на Ивана ===");
company.AssignEmployeeToOrder(order2.OrderId, ivan);
company.ChangeOrderStatus(order2.OrderId, OrderStatus.Completed);


Console.WriteLine();
Console.WriteLine("=== Заказы по статусам ===");
OrderStatus[] statuses = { OrderStatus.Created, OrderStatus.InProgress, OrderStatus.Completed, OrderStatus.Cancelled };
foreach (OrderStatus status in statuses)
{
    Console.WriteLine(status + ":");
    company.PrintOrders(company.GetOrdersByStatus(status));
}


Console.WriteLine();
Console.WriteLine("=== Заказы заказчика: " + oleg.GetFullName() + " ===");
company.PrintOrders(company.FindOrdersByCustomer(oleg));


Console.WriteLine();
company.PrintEmployeeReport();

The below script needs to be able to find the current output cell; this is an easy method to get it.

=== Участники ===
Сотрудник: Иван Петров | Должность: Менеджер | Обработано заказов: 0
Сотрудник: Анна Сидорова | Должность: Инженер | Обработано заказов: 0
Заказчик: Олег Смирнов | Телефон: +7-900-111-22-33
Заказчик: Мария Кузнецова | Телефон: +7-900-444-55-66

=== Созданные заказы ===
  Заказ #1 [Created] от 20.11.2026 19:14 | "Разработка сайта" | Клиент: Олег Смирнов | Ответственный: Не назначен
  Заказ #2 [Created] от 20.11.2026 19:14 | "Настройка сети" | Клиент: Мария Кузнецова | Ответственный: Не назначен
  Заказ #3 [Created] от 20.11.2026 19:14 | "Ремонт сервера" | Клиент: Олег Смирнов | Ответственный: Не назначен

=== Изменение статусов ===
[Уведомление для Иван Петров]: заказ #1 "Разработка сайта" сменил статус: Created -> InProgress
[Уведомление для Анна Сидорова]: заказ #2 "Настройка сети" сменил статус: Created -> InProgress
[Уведомление для Иван Петров]: заказ #1 "Разработка сайта" сменил статус: InProgress -> Completed
[Уведомление для Иван Петров]: заказ #3 "Ремонт серве